# Generowanie sztucznych danych

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

Dane na temat:
- wymagań dla dli i lux zostały zaciągnięte z pliku `dli_requirements.pdf`
- cen energii: https://www.tge.pl/energia-elektryczna-rdn
Wszystko to jest przyblizone i ma na celu stworzenie pewnego obrazu datasetu.

In [10]:
def generate_greenhouse_dataset(num_records=10000):
    plant_profiles = {
        'Tomato': {'dli_min': 10, 'dli_max': 30, 'max_lux': 800, 'co2_sens': 0.9},
        'Primula': {'dli_min': 4, 'dli_max': 22, 'max_lux': 400, 'co2_sens': 0.3},
        'Aglaonema': {'dli_min': 4, 'dli_max': 14, 'max_lux': 600, 'co2_sens': 0.6},
        'Orchid': {'dli_min': 2, 'dli_max': 10, 'max_lux': 200, 'co2_sens': 1.0}
    }

    data = []
    start_date = datetime(2026, 1, 1, 0, 0)

    for i in range(num_records):
        timestamp = start_date + timedelta(hours=i)
        hour = timestamp.hour
        
        # Wybór rośliny (w rzeczywistym CPS system rozpoznaje to kamerą)
        plant_type = random.choice(list(plant_profiles.keys()))
        profile = plant_profiles[plant_type]

        # Symulujemy etap życia rośliny: 0.1 (siewka), 1.0 (dojrzała)
        growth_stage = round(random.uniform(0.05, 1.0), 2)
        
        ndvi = round(min(0.9, (growth_stage * 0.8) + random.uniform(-0.05, 0.05)), 2)

        if 6 <= hour <= 19:
            # Sinusoida nasłonecznienia z losowym zachmurzeniem
            peak_factor = np.sin(np.pi * (hour - 6) / 13)
            sun_actual = max(0, (peak_factor * profile['max_lux']) + random.uniform(-150, 150))
            # Prognoza (forecast) z błędem +/- 15%
            sun_forecast = max(0, sun_actual * random.uniform(0.85, 1.15))
        else:
            sun_actual = 0
            sun_forecast = 0
       
        # Ceny energii - szczyty cenowe: rano (7-9) i wieczorem (17-20)
        if (7 <= hour <= 9) or (17 <= hour <= 20):
            grid_price = round(random.uniform(0.95, 1.45), 2)
            co2_intensity = random.randint(850, 1100) # g/kWh
        else:
            grid_price = round(random.uniform(0.40, 0.65), 2)
            co2_intensity = random.randint(450, 750)

        # CEL DLI (Zależy od gatunku i fazy wzrostu)
        target_dli = round(profile['dli_min'] + (profile['dli_max'] - profile['dli_min']) * growth_stage, 2)

        data.append({
            'Timestamp': timestamp,
            'Hour': hour,
            'Plant_Type': plant_type,
            'Growth_Stage': growth_stage,
            'NDVI_Camera_Index': ndvi,
            'Sunlight_Actual': round(sun_actual, 2),
            'Sunlight_Forecast': round(sun_forecast, 2),
            'Price_PLN_kWh': grid_price,
            'CO2_g_kWh': co2_intensity,
            'Target_DLI': target_dli,
            'Max_Threshold_Lux': profile['max_lux']
        })

    df = pd.DataFrame(data)
    df.to_csv('plants_dataset.csv', index=False)
    print(f"Wygenerowano pomyślnie {num_records} rekordów")

Generowanie datasetu dla 10_000 rekordów

In [11]:
generate_greenhouse_dataset()

Wygenerowano pomyślnie 10000 rekordów


In [13]:
df = pd.read_csv('plants_dataset.csv')
df.head(20)

,Timestamp,Hour,Plant_Type,Growth_Stage,NDVI_Camera_Index,Sunlight_Actual,Sunlight_Forecast,Price_PLN_kWh,CO2_g_kWh,Target_DLI,Max_Threshold_Lux
0,2026-01-01 00:00:00,0,Primula,0.67,0.53,0.00,0.00,0.45,578,16.06,400
1,2026-01-01 01:00:00,1,Aglaonema,0.46,0.39,0.00,0.00,0.59,664,8.60,600
2,2026-01-01 02:00:00,2,Primula,0.34,0.22,0.00,0.00,0.41,541,10.12,400
3,2026-01-01 03:00:00,3,Tomato,0.83,0.69,0.00,0.00,0.43,536,26.60,800
4,2026-01-01 04:00:00,4,Primula,0.30,0.26,0.00,0.00,0.51,614,9.40,400
5,2026-01-01 05:00:00,5,Orchid,0.86,0.73,0.00,0.00,0.46,658,8.88,200
6,2026-01-01 06:00:00,6,Orchid,0.18,0.18,0.00,0.00,0.48,746,3.44,200
7,2026-01-01 07:00:00,7,Orchid,0.68,0.51,31.49,28.42,1.07,856,7.44,200
8,2026-01-01 08:00:00,8,Tomato,0.74,0.61,461.67,416.63,1.39,1002,24.80,800
9,2026-01-01 09:00:00,9,Tomato,0.68,0.53,395.26,428.98,1.22,961,23.60,800


Opis kolumn:
1. Parametry Czasu i Identyfikacji
- `Timestamp`: Data i godzina pomiaru. Pozwala modelowi predykcyjnemu rozumieć cykle dobowe (np. że po godzinie 4 rano wkrótce nastąpi wschód słońca).
- `Hour`: Godzina (0–23). Ułatwia algorytmom szybkie filtrowanie godzin szczytu energetycznego.
- `Plant_Type`: Gatunek rośliny rozpoznany przez kamerę. Każdy typ ładuje inny profil wymagań biologicznych.

2. Parametry Biologiczne (Stan Rośliny)
- `Growth_Stage `(0.05 – 1.0): Bezjednostkowy wskaźnik fazy życia - 0.1 to siewka (małe zapotrzebowanie), 1.0 to roślina w pełni dojrzała (maksymalne zapotrzebowanie na światło).
- `NDVI_Camera_Index` (-1.0 do 1.0): Znormalizowany Różnicowy Wskaźnik Wegetacji.
0.1–0.3 (kiełkowanie), 0.7–0.9 (zdrowa gęsta biomasa). Jeśli ten wskaźnik nagle spadnie w danych, system CPS powinien wykryć chorobę lub stres.
- `Target_DLI` (mol/m²/d): Daily Light Integral. To „budżet fotonów”, który roślina musi otrzymać w ciągu doby. System sumuje światło słoneczne i sztuczne, by dobić do tej liczby przed końcem dnia

3. Parametry Fizyczne (Środowisko)
- `Sunlight_Actual` (μmol/m2s): Natężenie światła mierzone czujnikiem wewnątrz szklarni. Jednostka to mikromole fotonów na metr kwadratowy na sekundę (PPFD). To realna porcja energii trafiająca do liści w danej sekundzie.
- `Sunlight_Forecast` (μmol/m2s): Przewidywane nasłonecznienie pobrane z API pogodowego. Model predykcyjny porównuje tę wartość z `Sunlight_Actual`, by zdecydować, czy „czekać na darmowe słońce”.
- `Max_Threshold_Lux` (μmol/m2s): Fizyczna granica wytrzymałości rośliny. Jeśli Sunlight_Actual + lampy LED przekroczą tę wartość, roślina ulegnie uszkodzeniu (stres świetlny)

4. Parametry Ekonomiczne (Rynek i Ekologia)
- `Price_PLN_kWh` (PLN): Bieżąca cena energii elektrycznej na giełdzie (Rynek Dnia Następnego). Wartości: 0.40 (tania dolina nocna) do 1.50 (drogi szczyt wieczorny). System CPS stara się nie włączać lamp, gdy cena jest wysoka.
- `CO2_g_kWh` (g/kWh): Intensywność emisji dwutlenku węgla. Mówi o tym, ile gramów CO₂ wyemitowano do atmosfery, by wytworzyć 1 kWh energii w danej godzinie.
    - Niskie (np. 400): Dużo wiatru i słońca w sieci.
    - Wysokie (np. 1100): System oparty na węglu (brak wiatru).

In [15]:
df.columns

Index(['Timestamp', 'Hour', 'Plant_Type', 'Growth_Stage', 'NDVI_Camera_Index',
       'Sunlight_Actual', 'Sunlight_Forecast', 'Price_PLN_kWh', 'CO2_g_kWh',
       'Target_DLI', 'Max_Threshold_Lux'],
      dtype='str')

# Model co chcemy zbadać/osiągnać? 

1. Spójność biologiczna czyli dzienne zapotrzebowanie na DLI
- Wskaźnikiem byłoby odchylenie od celu DLI rośliny
- Cel to czy na koniec dnia suma dostarczonego światła zgadza się z Target_DLI
- Zweryfikowanie błędu: jeśli model zaraportuje cel osiągniety a widzmy fizycznie lub na kamerze, ze obumiera to widzimy, ze akceptuje błędne dane
2. Oszczędność szklarni
Wskaźnikiem jest średni koszt 1 mola sztucznego światła. Porównanie kostu w modelu prostym opisanym uprzednio z modelem predykcyjnym.
- Weryfikacja błędu: Model zaawansowany powinien mieć nizsyz koszt jednostkowy. Jeśli jest drozszy, oznacza to, ze jego predykcja nie ma sensu.
3. Ślad Węglowy 
- Wskaźnik: Sumaryczna emisja CO2 na cykl wzrostu.
- Cel: Zbadanie, o ile gramów udało się obniżyć emisję dzięki przesunięciu doświetlania na godziny o niskim CO2_g_kWh.

### Jak rozponać, ze model się myli?
1. Weryfikacja:  Kamera vs Czujnik
- Test: Pobierz Sunlight_Actual (czujnik) i porównaj z NDVI_Camera_Index oraz jasnością z kamery.
- Wynik: Jeśli czujnik pokazuje 0, a kamera widzi zielone, oświetlone liście — model musi odrzucić dane z czujnika jako fałszywe. Jeśli tego nie zrobi, model się myli.

2. Analiza Residuum (Residual Analysis)
System predykcyjny zawsze generuje prognozę (Sunlight_Forecast).
    - Test: Oblicz błąd prognozy e=Actual−Forecast.
    - Wynik: Jeśli błąd e stale rośnie lub ma stały zwrot (np. system zawsze zawyża słońce), mechanizm Continuous Learning powinien skorygować wagi. Jeśli model nadal ufa błędnej prognozie, oznacza to awarię logiki uczenia.

3. Wykrywanie Stresu 
- Test: Monitoruj korelację między natężeniem światła a tempem wzrostu (NDVI).
- Wynik: Jeśli system dostarcza światło (koszt rośnie), a NDVI spada lub stoi w miejscu, oznacza to przekroczenie Max_Threshold_Lux. Model "nie widzi" fizycznego cierpienia rośliny, skupiając się tylko na liczbach.